# Support Ticket Agent
This notebook demonstrates the flow of utlizing multiple agents to


* Summarize top user issue
* Identify the root causes
* Provide top recommendations



## Setup

In [1]:
pip install langdetect

In [2]:
import os
import kagglehub
import pandas as pd
import random
import asyncio
import re
import json
import google.generativeai as genai
from google.colab import userdata

# API setup

api_key = userdata.get('API_Agent')
genai.configure(api_key=api_key)

model = genai.GenerativeModel("gemini-2.5-flash-lite")

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


## 0. Download Dataset- Kaggle Customer Support on Twitter

In [3]:
path = kagglehub.dataset_download("thoughtvector/customer-support-on-twitter")
print("Path to dataset files:", path)

Using Colab cache for faster access to the 'customer-support-on-twitter' dataset.
Path to dataset files: /kaggle/input/customer-support-on-twitter


In [4]:
file_path = os.path.join(path + '/twcs', "twcs.csv")
df = pd.read_csv(file_path)
print(df.head())
print(df.shape)

   tweet_id   author_id  inbound                      created_at  \
0         1  sprintcare    False  Tue Oct 31 22:10:47 +0000 2017   
1         2      115712     True  Tue Oct 31 22:11:45 +0000 2017   
2         3      115712     True  Tue Oct 31 22:08:27 +0000 2017   
3         4  sprintcare    False  Tue Oct 31 21:54:49 +0000 2017   
4         5      115712     True  Tue Oct 31 21:49:35 +0000 2017   

                                                text response_tweet_id  \
0  @115712 I understand. I would like to assist y...                 2   
1      @sprintcare and how do you propose we do that               NaN   
2  @sprintcare I have sent several private messag...                 1   
3  @115712 Please send us a Private Message so th...                 3   
4                                 @sprintcare I did.                 4   

   in_response_to_tweet_id  
0                      3.0  
1                      1.0  
2                      4.0  
3                      5.0  
4

## Phase 1: Data Preparation

In [5]:
# Extract customer initial complaints
df_user_init_complaint = (
    df[(df.inbound == True) & df.in_response_to_tweet_id.isnull() & df.response_tweet_id.notnull()]
)

df_user_init_complaint["company_reply_tweet_id"] = (
    df_user_init_complaint["response_tweet_id"]
    .astype(str)
    .str.split(",")
    .str[0]
)

# Join to company
df_company = (
    df[df.inbound == False][['tweet_id', 'author_id']]
    .rename(columns={'tweet_id': 'company_reply_tweet_id', 'author_id': 'company'})
)
df_company['company_reply_tweet_id'] = df_company['company_reply_tweet_id'].astype(str)

df_user_init_complaint = pd.merge(df_user_init_complaint, df_company, on='company_reply_tweet_id', how='left')

#Select sample company- Airbnb
df_user_init_complaint = df_user_init_complaint[df_user_init_complaint.company == 'AirbnbHelp']

print(f"Total AirbnbHelp complaints: {len(df_user_init_complaint)}")

/tmp/ipykernel_8012/1844617743.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_user_init_complaint["company_reply_tweet_id"] = (


Total AirbnbHelp complaints: 5577


### Data cleaning

In [6]:
def clean_tweet(text):
    text = str(text)
    text = re.sub(r'@\w+', '', text)           # remove @mentions
    text = re.sub(r'http\S+|www\S+', '', text)  # remove URLs
    text = text.encode("ascii", "ignore").decode()  # remove non-ASCII
    text = text.lower()
    text = re.sub(r'!+', '!', text)
    text = re.sub(r'\?+', '?', text)
    text = re.sub(r'\.+', '.', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

df_main = df_user_init_complaint.copy()
df_main = df_main.dropna(subset=['text'])
df_main['clean_text'] = df_main['text'].astype(str).apply(clean_tweet)
df_main = df_main[df_main.clean_text.str.len() >= 20]

#Select 100 samples for demonstration
df_sample = df_main.sample(100, random_state=42)

In [7]:
#Filter out to only English user comments
from langdetect import detect
from langdetect.lang_detect_exception import LangDetectException

def is_english(text):
    try:
        return detect(str(text)) == "en"
    except LangDetectException:
        return False

df_sample = df_sample[df_sample["clean_text"].apply(is_english)].copy()
print(f"English complaints: {len(df_sample)}")

English complaints: 95


## Phase 2: Clustering
Instead of asking LLM to summarize issues into clusters, which consumes lots of tokens, I firstly
- Convert sentences into vectors using SentenceTransformer
- Then cluster the complaints into different clusters using K-means clustering technique.

In [8]:
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
texts = df_sample["clean_text"].tolist()
embeddings = embedding_model.encode(texts, show_progress_bar=True)
print("Embedding shape:", embeddings.shape)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/3 [00:00<?, ?it/s]

Embedding shape: (95, 384)


In [9]:
#Find the best cluster number and its Silhouette score
cluster_score = []
for k in range(5, 30):
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=5)
    cluster_labels = kmeans.fit_predict(embeddings)
    score = silhouette_score(embeddings, cluster_labels)
    cluster_score.append({'k': k, "silhouette_score": score})

score_df = pd.DataFrame(cluster_score)
best_k = int(score_df.loc[score_df.idxmax()['silhouette_score']]['k'])
best_score = score_df.loc[score_df.idxmax()['silhouette_score']]['silhouette_score']
print(f'Best cluster: {best_k}  |  Silhouette score: {best_score:.4f}')

Best cluster: 29  |  Silhouette score: 0.0506


In [10]:
#Cluster issues into clusters
kmeans = KMeans(n_clusters=best_k, random_state=42)
df_sample['cluster_id'] = kmeans.fit_predict(embeddings)
print("Cluster distribution:")
print(df_sample["cluster_id"].value_counts().sort_index())

Cluster distribution:
cluster_id
0      2
1      3
2     10
3      3
4      8
5      2
6      7
7      3
8      8
9      3
10     3
11     2
12     7
13     5
14     2
15     5
16     2
17     2
18     2
19     2
20     2
21     1
22     1
23     3
24     2
25     2
26     1
27     1
28     1
Name: count, dtype: int64


### Identify top keywords from each cluster

In [11]:
from sklearn.feature_extraction.text import TfidfVectorizer

def get_cluster_keywords(texts, top_n=6):
    vectorizer = TfidfVectorizer(
        stop_words="english",
        ngram_range=(1, 3),
        max_features=1000
    )
    X = vectorizer.fit_transform(texts)
    scores = X.sum(axis=0).A1
    terms = vectorizer.get_feature_names_out()
    top_idx = scores.argsort()[-top_n:][::-1]
    return [terms[i] for i in top_idx]

cluster_keywords_result = []

for cluster_id in sorted(df_sample.cluster_id.unique()):

    cluster_texts = df_sample[df_sample.cluster_id == cluster_id]['clean_text']
    keywords = get_cluster_keywords(cluster_texts)
    cluster_keywords_result.append({
        "cluster_id": cluster_id,
        "keyword_label": " / ".join(keywords[:3]),
        "top_keywords": keywords,
        "cluster_size": len(cluster_texts)
    })

cluster_keywords_df = pd.DataFrame(cluster_keywords_result)

df_sample = pd.merge(
    df_sample,
    cluster_keywords_df[['cluster_id', 'keyword_label', 'top_keywords', 'cluster_size']],
    on='cluster_id', how='left'
)

print(cluster_keywords_df)

    cluster_id                                      keyword_label  \
0            0             place / suitable place stay / suitable   
1            1                        request / inquiry / listing   
2            2                     host / reservation / cancelled   
3            3                         dm / send / follow send dm   
4            4                                   host / help / ve   
5            5                                feel / robbed / use   
6            6                               host / airbnb / good   
7            7                             booking / request / hi   
8            8              waiting / resolved / problem resolved   
9            9                         price / date / clear dates   
10          10                               host / help / theres   
11          11                 book / using phone service / using   
12          12                            airbnb / terrible / hot   
13          13     trip / reservat

## Phase 3: Async Map-Reduce Agent Pipeline
This code is used to minimize the tokens burnt by using parallelized structure.

**Architecture:**
```
All clusters
     │
     ├── Chunk 1 (3 clusters) ──→ MAP Agent ──┐
     ├── Chunk 2 (3 clusters) ──→ MAP Agent ──┼──→ REDUCE Agent ──→ Final Report
     └── Chunk N (3 clusters) ──→ MAP Agent ──┘
            ↑ staggered + semaphore controlled
```
- Each MAP agent sees only a small chunk → no token limit
- All MAP calls fire simultaneously → faster
- REDUCE agent sees only compressed summaries → tiny payload

In [12]:
# Retrieve cluster examples

def get_cluster_examples(df, cluster_id, top_n=3):
    """Return top_n representative complaints, truncated to 150 chars each."""
    return (
        df[df.cluster_id == cluster_id]['clean_text']
        .astype(str)
        .head(top_n)
        .apply(lambda t: t[:150])
        .tolist()
    )


In [17]:
# ── Reusable Map-Reduce Engine ────────────────────────────────────────────────
# Credit: Claude

async def map_reduce(
    items           : list[dict],   # any data
    map_prompt_fn,                  # function(chunk) → prompt string
    reduce_prompt_fn,               # function(summaries) → prompt string
    chunk_size      : int = 3,
    concurrent      : int = 1,
    chunk_gap       : int = 8,
    max_retries     : int = 3,
) -> tuple[list[dict], dict]:
    """
    Generic Map-Reduce engine for any LLM analysis task.

    Usage:
        map_results, final = await map_reduce(
            items            = my_data,
            map_prompt_fn    = lambda chunk: f"Analyze this: {chunk}",
            reduce_prompt_fn = lambda summaries: f"Synthesize: {summaries}",
        )
    """

    # ── MAP ───────────────────────────────────────────────────────────────────
    chunks    = [items[i:i+chunk_size] for i in range(0, len(items), chunk_size)]
    semaphore = asyncio.Semaphore(concurrent)
    print(f"MAP: {len(items)} items → {len(chunks)} chunks of ~{chunk_size}")

    async def _call_with_retry(prompt, label):
        for attempt in range(max_retries):
            try:
                response = await asyncio.to_thread(model.generate_content, prompt)

                if not response.text or not response.text.strip():
                    raise ValueError("Empty response from model")

                raw = response.text.replace("```json","").replace("```","").strip()
                return json.loads(raw)
            except Exception as e:
                wait = (2 ** attempt) + random.uniform(0, 1)
                print(f"  [{label}] Attempt {attempt+1} failed: {e}. Retrying in {wait:.1f}s...")
                await asyncio.sleep(wait)
        print(f"  [{label}] All retries failed")
        return {}

    async def _map_chunk(chunk, idx):
        await asyncio.sleep(idx * chunk_gap)   # stagger to avoid quota burst
        async with semaphore:
            prompt = map_prompt_fn(chunk)
            result = await _call_with_retry(prompt, f"Chunk {idx}")
            return result if isinstance(result, list) else [result]

    chunk_results = await asyncio.gather(*[_map_chunk(c, i) for i, c in enumerate(chunks)])
    map_results   = [item for sublist in chunk_results for item in sublist]
    print(f"MAP complete: {len(map_results)} results")

    # ── REDUCE ────────────────────────────────────────────────────────────────
    print(f"REDUCE: synthesizing {len(map_results)} results...")
    reduce_prompt = reduce_prompt_fn(map_results)
    final_result  = await _call_with_retry(reduce_prompt, "Reduce")
    print("REDUCE complete ✓")

    return map_results, final_result


In [18]:
# ── Support Ticket Pipeline ───────────────────────────────────────────────────

async def run_pipeline(df, cluster_keywords_df):
    """Run the full Map-Reduce agent pipeline for support ticket analysis."""

    # Prepare items — compact payload per cluster
    items = [
        {
            "cluster_id"  : int(row["cluster_id"]),
            "cluster_size": row["cluster_size"],
            "keywords"    : row["top_keywords"][:6],
            "examples"    : get_cluster_examples(df, int(row["cluster_id"]), top_n=3)
        }
        for _, row in cluster_keywords_df.iterrows()
    ]

    # MAP prompt — what each chunk agent sees
    def map_prompt(chunk):
        return f"""
        You are a customer support analyst. Analyze these complaint clusters.

        Clusters:
        {json.dumps(chunk, ensure_ascii=False, indent=2)}

        Return ONLY a JSON array — no explanation, no markdown fences:
        [
          {{
            "cluster_id": 0,
            "issue_label": "short label (3-5 words)",
            "issue_description": "one sentence",
            "likely_root_cause": "one sentence",
            "business_severity": "Low | Medium | High",
            "recommended_action": "one sentence",
            "customer_pain_point": "one sentence"
          }}
        ]
        """

    # REDUCE prompt — what the final synthesis agent sees
    def reduce_prompt(summaries):
        return f"""
        You are a senior customer support analyst writing an executive report.

        Below are analyses of {len(summaries)} customer complaint clusters.
        Synthesize them into a concise report.

        Cluster analyses:
        {json.dumps(summaries, ensure_ascii=False, indent=2)}

        Return ONLY a JSON object — no explanation, no markdown fences:
        {{
          "top_issues": [
            {{"rank": 1, "issue": "...", "affected_clusters": [...], "severity": "High | Medium | Low"}}
          ],
          "root_causes": [
            {{"rank": 1, "cause": "...", "related_issues": [...]}}
          ],
          "recommendations": [
            {{"priority": 1, "action": "...", "expected_impact": "...", "effort": "Low | Medium | High"}}
          ],
          "executive_summary": "2-3 sentence overall summary"
        }}
        """

    return await map_reduce(
        items            = items,
        map_prompt_fn    = map_prompt,
        reduce_prompt_fn = reduce_prompt,
        chunk_size       = 3,
        concurrent       = 1,
        chunk_gap        = 5,
    )


# Run pipeline code
map_summaries, final_report = await run_pipeline(df_sample, cluster_keywords_df)


MAP: 29 items → 10 chunks of ~3
MAP complete: 29 results
REDUCE: synthesizing 29 results...
REDUCE complete ✓


In [19]:
print("Key starts with:", api_key[:15])
print("Key length:", len(api_key))

Key starts with: AIzaSyCVxwJxFwi
Key length: 39


## Results

In [20]:
# Per-cluster results
cluster_agent_df = pd.DataFrame(map_summaries)
print("=== CLUSTER-LEVEL RESULTS ===")
print(cluster_agent_df[['cluster_id', 'issue_label', 'business_severity', 'likely_root_cause']].to_string(index=False))

=== CLUSTER-LEVEL RESULTS ===
 cluster_id                               issue_label business_severity                                                                                                                                                                          likely_root_cause
          0     Unsuitable Accommodation/Cleaning Fee            Medium                                                               Inconsistent property standards and potentially inflated cleaning fees are leading to guest dissatisfaction and abandonment.
          1                    Inquiry/Request Issues            Medium                                 Issues with listing visibility, platform functionality, or guest communication regarding mandatory procedures are hindering bookings and causing friction.
          2 Host Cancellations Impacting Reservations              High                                                         A lack of host accountability or support for unexpected cancellat

In [21]:
# Final report
print("=== EXECUTIVE SUMMARY ===")
print(final_report.get('executive_summary', 'N/A'))

print("\n=== TOP ISSUES ===")
for issue in final_report.get('top_issues', []):
    print(f"  #{issue['rank']} [{issue['severity']}] {issue['issue']}")

print("\n=== ROOT CAUSES ===")
for cause in final_report.get('root_causes', []):
    print(f"  #{cause['rank']} {cause['cause']}")

print("\n=== RECOMMENDATIONS ===")
for rec in final_report.get('recommendations', []):
    print(f"  P{rec['priority']} [{rec['effort']} effort] {rec['action']}")
    print(f"      Impact: {rec['expected_impact']}")

=== EXECUTIVE SUMMARY ===
Analysis of 29 customer complaint clusters reveals critical issues in guest safety, booking reliability due to host cancellations, and financial/account access problems. These high-severity issues stem from inadequate security, operational inefficiencies, and inconsistent quality control, demanding immediate and comprehensive action to restore customer trust and satisfaction.

=== TOP ISSUES ===
  #1 [High] Guest Safety and Security Incidents
  #2 [High] Host Cancellations Impacting Reservations
  #3 [High] Cancellation & Refund Delays
  #4 [High] Account access and verification
  #5 [High] Unauthorized Apartment Bookings
  #6 [High] Extremely Negative Customer Experience
  #7 [High] Urgent issue, no reply
  #8 [High] Reservation Cancellation Policy Confusion
  #9 [High] Host Support and Dispute Resolution
  #10 [High] Booking Discrepancies and Issues

=== ROOT CAUSES ===
  #1 Inadequate security measures and incident response protocols.
  #2 Operational ineff

In [22]:
# Save outputs
from google.colab import drive
drive.mount('/content/drive')

import os
save_dir = '/content/drive/MyDrive/Gen-AI/support-agent'
os.makedirs(save_dir, exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [25]:
df_sample.to_csv(f'{save_dir}/df_sample.csv', index=False)

cluster_agent_df.to_csv(f'{save_dir}/cluster_analysis.csv', index=False)

with open(f'{save_dir}/final_report.json', 'w') as f:
    json.dump(final_report, f, indent=2)

print(f'Saved to {save_dir}')

Saved to /content/drive/MyDrive/Gen-AI/support-agent
